## NLP in PyTorch

### Importing Libraries

In [2]:
import torchvision
import torch
from torchvision import transforms
from PIL import Image
import math
import os
import shutil
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
from torchvision import models
import torch.optim as optim
from torch.optim import Adam
import pandas as pd


In [3]:
# Dowbloading the dataset
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ferno2/training1600000processednoemoticoncsv")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/training1600000processednoemoticoncsv


In [4]:
tweetsPDF=pd.read_csv("/root/.cache/kagglehub/datasets/ferno2/training1600000processednoemoticoncsv/versions/1/training.1600000.processed.noemoticon.csv",engine="python",header=None,encoding="ISO-8859-1",
    )

In [5]:
tweetsPDF.head(5)

,0,1,2,3,4,5
0,0,1467810369,Mon Apr 06 22:19:45 PDT 2009,NO_QUERY,_TheSpecialOne_,"@switchfoot http://twitpic.com/2y1zl - Awww, t..."
1,0,1467810672,Mon Apr 06 22:19:49 PDT 2009,NO_QUERY,scotthamilton,is upset that he can't update his Facebook by ...
2,0,1467810917,Mon Apr 06 22:19:53 PDT 2009,NO_QUERY,mattycus,@Kenichan I dived many times for the ball. Man...
3,0,1467811184,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,ElleCTF,my whole body feels itchy and like its on fire
4,0,1467811193,Mon Apr 06 22:19:57 PDT 2009,NO_QUERY,Karoli,"@nationwideclass no, it's not behaving at all...."


In [6]:
tweetsPDF[0].value_counts()

,count
0,
0,800000
4,800000


In [7]:
tweetsPDF["sentiment_cat"]=tweetsPDF[0].astype("category")

In [8]:
tweetsPDF.to_csv("tweets.csv",index=False,header=None)

In [9]:
tweetsPDF.sample(10000).to_csv("train-processed-sample.csv",header=None,index=False)

Refer to field parameter table on this page https://docs.pytorch.org/text/0.8.1/data.html

In [10]:
from torchtext import data

Old Version

In [ ]:
LABEL=data.LabelField()
TWEET=data.Field(tokenize="spacy",lower=True)
fields=[("score",None),("id",None),("date",None),("query",None),("name",None),("tweet",TWEET),("label",LABEL)]
twitterDataset=data.TabularDataset(path="train-processed-sample.csv",format="csv",skip_header=False,fields=fields)
(train,test,val)=twitterDataset.split(split_ratio=[0.8,0.1,0.1])
vocab_size=20000
TWEET.build_vocab(train,max_size=vocab_size)
print(len(TWEET.vocab))
print(TWEET.vocab.freqs.most_common(10))
train_iter,test_iter,val_iter=data.BucketIterator.splits((train,test,val),batch_size=32,device="cpu")




New Version

In [26]:
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader, random_split
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torch.nn.utils.rnn import pad_sequence

# Load CSV and rename columns
df = pd.read_csv("train-processed-sample.csv", header=None)
df.columns = ["score", "id", "date", "query", "name", "tweet", "label"]

# Tokenizer
tokenizer = get_tokenizer("spacy", language="en_core_web_sm")

# Build vocab from tweets
def yield_tokens(data_iter):
    for text in data_iter:
        yield tokenizer(text)

vocab_size = 20000
vocab = build_vocab_from_iterator(yield_tokens(df["tweet"]), specials=["<unk>"], max_tokens=vocab_size)
vocab.set_default_index(vocab["<unk>"])

# Label mapping (binary example)
label_map = {"positive": 1, "negative": 0}

# Pipelines
def text_pipeline(text):
    return vocab(tokenizer(text))

def label_pipeline(label):
    return label_map[label]

# Custom Dataset
class TwitterDataset(Dataset):
    def __init__(self, dataframe):
        self.data = dataframe

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tweet = self.data.iloc[idx]["tweet"]
        label = self.data.iloc[idx]["label"]
        return torch.tensor(text_pipeline(tweet), dtype=torch.int64), torch.tensor(label_pipeline(label), dtype=torch.int64)

# Create full dataset
full_dataset = TwitterDataset(df)

# Split into train, test, val
train_size = int(0.8 * len(full_dataset))
test_size = int(0.1 * len(full_dataset))
val_size = len(full_dataset) - train_size - test_size
train_dataset, test_dataset, val_dataset = random_split(full_dataset, [train_size, test_size, val_size])

# Collate function for padding
def collate_batch(batch):
    tweets, labels = zip(*batch)
    tweets_padded = pad_sequence(tweets, batch_first=True, padding_value=vocab["<unk>"])
    labels = torch.stack(labels)
    return tweets_padded, labels

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_dataset, batch_size=32, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_batch)

# Inspect vocab
print("Vocab size:", len(vocab))
print("Top 10 tokens:", vocab.get_itos()[:10])


Vocab size: 20000
Top 10 tokens: ['<unk>', '!', '.', 'I', ' ', 'to', 'the', ',', 'a', 'i']


### Creating Model

In [65]:
import torch.nn as nn
class OurModel(nn.Module):
    def __init__(self,hidden_size,embedding_dim,vocab_size):
        super(OurModel,self).__init__()
        self.embed=nn.Embedding(vocab_size,embedding_dim)
        self.encoder=nn.LSTM(input_size=embedding_dim,hidden_size=hidden_size,num_layers=1)
        self.predictor=nn.Linear(hidden_size,2)
    def forward(self,seq):
        output,(hidden,_)=self.encoder(self.embed(seq))
       # hidden = hidden[-1]  # Last layer's hidden state

        preds=self.predictor(hidden.squeeze(0))
        return preds
model=OurModel(100,300,20002)
model.to("cpu")

OurModel(
  (embed): Embedding(20002, 300)
  (encoder): LSTM(300, 100)
  (predictor): Linear(in_features=100, out_features=2, bias=True)
)

In [66]:
optimizer=optim.Adam(model.parameters(),lr=2e-2)
criterion=nn.CrossEntropyLoss()


In [67]:
def train(epochs,model,optimizer,criterion,train_loader,val_loader):
  for epoch in range(1,epochs+1):
    training_loss=0
    valid_loss=0
    model.train()
    for batch_idx,batch in enumerate(train_loader):
      optimizer.zero_grad()
      output=model(batch.tweet)
      loss=criterion(output,batch.label)
      loss.backward()
      optimizer.step()
      training_loss=training_loss.item()*batch.tweet.size(0)
    training_loss=training_loss/len(train_loader.dataset)
    model.eval()
    for batch_idx,batch in enumerate(val_loader):
      output=model(batch.tweet)
      loss=criterion(output,batch.label)
      valid_loss=valid_loss.item()*batch.tweet.size(0)
    valid_loss=valid_loss/len(val_loader.dataset)
    print("Epoch: {} \tTraining Loss: {} \tValidation Loss: {}".format(epoch,training_loss,valid_loss))

In [72]:
def classify_tweet(tweet, model, vocab):
    categories = {0: "Negative", 1: "Positive"}

    tokenizer = get_tokenizer("spacy", language="en_core_web_sm")
    text_pipeline = lambda x: vocab(tokenizer(x))

    tweet_tensor = torch.tensor(text_pipeline(tweet), dtype=torch.int64).unsqueeze(0)  # shape: (1, seq_len)

    with torch.no_grad():
        output = model(tweet_tensor)  # shape: (1, num_classes)
        print("output shape",output.shape)
        # Average over the sequence dimension (dim=0), then argmax over classes
        pooled_output = output.mean(dim=0)  # shape: (2,)
        prediction = pooled_output.argmax().item()

    return categories[prediction]


In [74]:
classify_tweet("I love this",model,vocab)

output shape torch.Size([3, 2])


'Positive'

### Data Augmenation

In [80]:
# Random Insertion
from nltk.corpus import wordnet

def get_synonyms(word):
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonym = lemma.name().replace("_", " ").lower()
            if synonym != word:
                synonyms.add(synonym)
    return list(synonyms)
import nltk
nltk.download('wordnet')

import random
from random import randrange

def random_insertion(sentence, n):
    words = sentence.split()
    for _ in range(n):
        word = random.choice(words)
        synonyms = get_synonyms(word)
        if synonyms:
            new_synonym = random.choice(synonyms)
            insert_pos = randrange(len(words) + 1)
            words.insert(insert_pos, new_synonym)
    return " ".join(words)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [81]:
random_insertion("The cat sat on the mat",6)

'The along cat posture sat on the mat'

In [83]:
## Random Deletion
s1="The cat sat on the mat"
def random_deletion(words, p=0.5):
    if len(words) == 1:
        return words
    remaining=list(filter(lambda x:random.uniform(0,1)>p,words))
    if len(remaining)==0:
        return [random.choice(words)]
    else:
        return remaining

In [86]:
random_deletion(s1, p=0.5)

['T', 'h', 'e', ' ', 't', 't', ' ', ' ', 't', 'h', ' ', 'a']

In [90]:
import random

def random_swap(sentence, n=1):
    words = sentence.split()
    length = len(words)

    for _ in range(n):
        if length < 2:
            break  # Can't swap if fewer than 2 words
        idx1, idx2 = random.sample(range(length), 2)
        words[idx1], words[idx2] = words[idx2], words[idx1]

    return " ".join(words)


In [91]:
random_swap(s1)

'The cat mat on the sat'

### Back Translation

In [93]:
pip install googletrans

In [102]:
import asyncio
from googletrans import Translator

translator = Translator()
sentences = ["The cat sat on the mat"]

async def translate_sentences(sentences):
    translation_fr = [await translator.translate(s, dest='fr') for s in sentences]
    fr_text = [t.text for t in translation_fr]

    translation_en = [await translator.translate(s, dest='en') for s in fr_text]
    en_text = [t.text for t in translation_en]

    print("Original:", sentences)
    print("French:", fr_text)
    print("Back to English:", en_text)

await translate_sentences(sentences)


<frozen abc>:123: RuntimeWarning: coroutine 'Translator.translate' was never awaited


Original: ['The cat sat on the mat']
French: ['Le chat était assis sur le tapis']
Back to English: ['The cat was sitting on the carpet']


In [99]:
translation_fr

<coroutine object Translator.translate at 0x7c019b9fb6e0>